# 13 — WC 2026 Simulation

48-team format (12 groups × 4, top 2 + 8 best 3rds → R32). Official group draw from December 2025.

Uses regression-Dixon-Coles (notebook 10) as the goal model since it handles cold-start 2026 squads better than plain DC. Feature inputs: current Elo, FIFA points, latest squad market value.

In [1]:
import sys
sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from src.elo import EloModel
from src.regression_dc import RegressionDixonColesModel
from src.bracket import simulate_tournament_48, WC_2026_GROUPS

## Fit regression-DC on full history through 2025

In [2]:
df_all = pd.read_csv('../data/processed/matches_competitive.csv', parse_dates=['date']).dropna(subset=['home_score','away_score'])
df_sv  = pd.read_csv('../data/processed/matches_with_squad_value.csv', parse_dates=['date'])
elo = EloModel()
elo_enriched = elo.fit(df_all)
df = df_sv.merge(elo_enriched[['date','home_team','away_team','home_elo_pre','away_elo_pre']],
                  on=['date','home_team','away_team'], how='left').dropna(subset=['home_elo_pre','away_elo_pre'])

FLOOR = 1e5
df['home_elo']  = df['home_elo_pre']/100.0; df['away_elo']  = df['away_elo_pre']/100.0
df['home_logv'] = np.log10(df['home_top_n_value_eur'].clip(lower=FLOOR))
df['away_logv'] = np.log10(df['away_top_n_value_eur'].clip(lower=FLOOR))
df['home_fpts'] = df['home_fifa_points'].fillna(0)/1000.0
df['away_fpts'] = df['away_fifa_points'].fillna(0)/1000.0

# Train on all matches with all features (uses real-Fjelstul-roster values on WC matches, pool on others)
RDC_FEAT = ['elo','logv','fpts']
needed = [f'{s}_{c}' for s in ('home','away') for c in RDC_FEAT] + ['home_score','away_score']
train = df.dropna(subset=needed)
print(f'Training on {len(train):,} matches through {train["date"].max():%Y-%m-%d}')
rdc = RegressionDixonColesModel(RDC_FEAT, xi=0.00038).fit(train)
print('Reg-DC coefficients:')
print(rdc.coefficients().round(4).to_string(index=False))

Training on 13,067 matches through 2026-03-31
Reg-DC coefficients:
feature  w_attack  w_defense
    elo    0.2733     0.2477
   logv    0.0386     0.1097
   fpts    0.1679     0.1995


## Bridge to DC-shaped interface

The bracket simulator was built for the plain `DixonColesModel`. Reg-DC has a different per-team representation (alpha/beta computed from features at predict time), so we'll wrap it in a thin shim that exposes the same `attack`/`defense`/`_rates`/`score_matrix` interface using each team's *latest available features*.

In [3]:
from scipy.stats import poisson

class RegDCShim:
    '''Wraps a fitted RegressionDixonColesModel and a per-team feature snapshot.
    Exposes the subset of the DixonColesModel API that bracket.py uses.'''
    def __init__(self, rdc, team_features: dict[str, np.ndarray]):
        self.rdc = rdc
        self.team_features = team_features  # team -> raw feature vector
        # Pre-compute alpha/beta for each team (using rdc's centering)
        self.attack = {}
        self.defense = {}
        for t, x in team_features.items():
            x_c = x - rdc.feature_mean_
            self.attack[t]  = float(x_c @ rdc.w_a)
            self.defense[t] = float(x_c @ rdc.w_b)
        self.home_adv = rdc.home_adv
        self.rho = rdc.rho
        self.max_goals = rdc.max_goals

    def _rates(self, home, away, neutral=False):
        a_h = self.attack[home]; d_h = self.defense[home]
        a_a = self.attack[away]; d_a = self.defense[away]
        gamma = 0.0 if neutral else self.home_adv
        lam_h = float(np.exp(a_h - d_a + gamma))
        lam_a = float(np.exp(a_a - d_h))
        return lam_h, lam_a

    def score_matrix(self, home, away, neutral=False):
        lam_h, lam_a = self._rates(home, away, neutral)
        x = np.arange(self.max_goals + 1)
        px = poisson.pmf(x, lam_h); py = poisson.pmf(x, lam_a)
        m = np.outer(px, py).copy()
        m[0, 0] *= 1 - lam_h * lam_a * self.rho
        m[0, 1] *= 1 + lam_h * self.rho
        m[1, 0] *= 1 + lam_a * self.rho
        m[1, 1] *= 1 - self.rho
        return m

# Build per-team latest feature snapshot — use each team's last appearance in df
team_features = {}
for t in sum(WC_2026_GROUPS.values(), []):
    h_recent = df[df['home_team']==t].sort_values('date').tail(1)
    a_recent = df[df['away_team']==t].sort_values('date').tail(1)
    if not h_recent.empty and (a_recent.empty or h_recent['date'].iloc[0] >= a_recent['date'].iloc[0]):
        team_features[t] = h_recent[['home_elo','home_logv','home_fpts']].iloc[0].to_numpy(dtype=float)
    elif not a_recent.empty:
        team_features[t] = a_recent[['away_elo','away_logv','away_fpts']].iloc[0].to_numpy(dtype=float)
    else:
        team_features[t] = np.array([rdc.feature_mean_[i] for i in range(3)])  # fallback to average

shim = RegDCShim(rdc, team_features)
ratings = pd.DataFrame({
    'team': list(team_features.keys()),
    'attack': [shim.attack[t] for t in team_features],
    'defense': [shim.defense[t] for t in team_features],
    'group': [g for g, ts in WC_2026_GROUPS.items() for t in ts if t in team_features],
}).assign(skill=lambda d: d['attack']+d['defense']).sort_values('skill', ascending=False)
print('Top 12 by total skill (attack + defense):')
print(ratings.head(12).round(3).to_string(index=False))
print()
print('Bottom 6:')
print(ratings.tail(6).round(3).to_string(index=False))

Top 12 by total skill (attack + defense):
       team  attack  defense group  skill
      Spain   1.315    1.428     H  2.743
    England   1.025    1.161     L  2.186
     France   1.007    1.146     I  2.154
  Argentina   0.961    1.088     J  2.049
   Portugal   0.812    0.954     K  1.765
Netherlands   0.792    0.930     F  1.722
      Japan   0.749    0.855     F  1.604
    Germany   0.696    0.842     E  1.538
    Morocco   0.708    0.826     C  1.534
     Brazil   0.679    0.840     C  1.519
    Belgium   0.648    0.781     G  1.429
    Croatia   0.654    0.768     L  1.421

Bottom 6:
                  team  attack  defense group  skill
           Ivory Coast   0.315    0.195     E  0.510
              Paraguay   0.195    0.308     D  0.502
               Curaçao   0.245    0.120     E  0.365
              DR Congo   0.237    0.122     K  0.359
            Cape Verde   0.088   -0.018     H  0.070
Bosnia and Herzegovina  -0.096   -0.184     B -0.280


## Run 48-team Monte Carlo simulation

In [4]:
results = simulate_tournament_48(WC_2026_GROUPS, shim, n_sims=5000, n_group_sims=2000, seed=0)
pd.options.display.float_format = '{:.3f}'.format
print('Top 16 by P(winner):')
print(results.head(16).to_string(index=False))
print()
print('Bottom 8 (lowest P(R32)):')
print(results.sort_values('p_R32').head(8).to_string(index=False))

Top 16 by P(winner):
       team group  p_R32  p_R16  p_QF  p_SF  p_final  p_winner
      Spain     H  1.000  0.912 0.794 0.656    0.545     0.407
    England     L  0.993  0.660 0.452 0.271    0.191     0.098
     France     I  0.987  0.680 0.439 0.262    0.181     0.094
  Argentina     J  0.977  0.708 0.398 0.233    0.153     0.074
Netherlands     F  0.967  0.664 0.424 0.194    0.096     0.040
      Japan     F  0.945  0.626 0.388 0.187    0.089     0.034
   Portugal     K  0.986  0.600 0.279 0.133    0.074     0.029
    Morocco     C  0.911  0.549 0.334 0.183    0.067     0.027
     Brazil     C  0.912  0.556 0.344 0.191    0.063     0.025
    Germany     E  0.980  0.622 0.321 0.110    0.055     0.022
Switzerland     B  0.993  0.556 0.275 0.137    0.040     0.014
    Belgium     G  0.915  0.562 0.206 0.103    0.039     0.014
South Korea     A  0.927  0.468 0.254 0.134    0.043     0.012
     Mexico     A  0.926  0.473 0.252 0.132    0.036     0.012
       Iran     G  0.884  0.516 0.

## Group-by-group breakdown

In [5]:
for g in sorted(WC_2026_GROUPS):
    sub = results[results['group']==g].sort_values('p_R16', ascending=False)
    print(f'\nGroup {g}:')
    print(sub[['team','p_R16','p_QF','p_SF','p_final','p_winner']].to_string(index=False))


Group A:
          team  p_R16  p_QF  p_SF  p_final  p_winner
        Mexico  0.473 0.252 0.132    0.036     0.012
   South Korea  0.468 0.254 0.134    0.043     0.012
  South Africa  0.078 0.020 0.005    0.001     0.000
Czech Republic  0.076 0.026 0.008    0.001     0.000

Group B:
                  team  p_R16  p_QF  p_SF  p_final  p_winner
           Switzerland  0.556 0.275 0.137    0.040     0.014
                Canada  0.481 0.209 0.095    0.025     0.009
                 Qatar  0.296 0.087 0.028    0.005     0.002
Bosnia and Herzegovina  0.004 0.000 0.000    0.000     0.000

Group C:
    team  p_R16  p_QF  p_SF  p_final  p_winner
  Brazil  0.556 0.344 0.191    0.063     0.025
 Morocco  0.549 0.334 0.183    0.067     0.027
Scotland  0.221 0.083 0.026    0.007     0.001
   Haiti  0.190 0.070 0.024    0.005     0.001

Group D:
         team  p_R16  p_QF  p_SF  p_final  p_winner
    Australia  0.465 0.223 0.099    0.024     0.006
United States  0.458 0.215 0.096    0.026     0.009